# CIGRE MV Powerflow Solver Test

## Run simulation

In [ ]:
import requests
import glob


def download_grid_data(name, url):
    with open(name, "wb") as out_file:
        content = requests.get(url, stream=True).content
        out_file.write(content)


url = "https://raw.githubusercontent.com/dpsim-simulator/cim-grid-data/master/CIGRE_MV/NEPLAN/CIGRE_MV_no_tapchanger_With_LoadFlow_Results/Rootnet_FULL_NE_06J16h"
filename = "CIGRE-MV"
download_grid_data(filename + "_EQ.xml", url + "_EQ.xml")
download_grid_data(filename + "_TP.xml", url + "_TP.xml")
download_grid_data(filename + "_SV.xml", url + "_SV.xml")

files = glob.glob(filename + "_*.xml")
print(files)

In [ ]:
import dpsimpy

In [ ]:
name = "CIGRE_MV"
reader = dpsimpy.CIMReader(name)
system = reader.loadCIM(
    50, files, dpsimpy.Domain.SP, dpsimpy.PhaseType.Single, dpsimpy.GeneratorType.PVNode
)

In [ ]:
# system.render_to_file('CIGRE_MV.svg')
system

This feeder has no tap changer, so its reference load flow shows a realistic voltage drop of about 10 percent, which exceeds the solver's default zone-consistency tolerance.

In [ ]:
sim = dpsimpy.Simulation(name)
sim.set_system(system)
sim.set_domain(dpsimpy.Domain.SP)
sim.set_solver(dpsimpy.Solver.NRP)
sim.set_pf_solver_base_voltage_loose_tolerance(0.15)
sim.set_time_step(1)
sim.set_final_time(10)

logger = dpsimpy.Logger(name)
for node in system.nodes:
    logger.log_attribute(node.name() + ".V", "v", node)
sim.add_logger(logger)

In [ ]:
sim.start()

In [ ]:
sim.next()

In [ ]:
sim.get_idobj_attr(
    "LOAD-I-10", "P_pu"
).get()  # The get method provides a reference to the underlying value

In [ ]:
sim.get_idobj_attr(
    "N10", "v"
)  # This also works because the Attribute returned by get_idobj_attr is printable

In [ ]:
sim.get_idobj_attr("L1-2", "i_intf")

In [ ]:
sim.get_idobj_attr("LOAD-I-10", "P_pu").set(
    0.00128
)  # Setting values can be done using an attribute's set function

In [ ]:
system.list_idobjects()